# Ball outside the pitch (clause a)

Tests clause a of the draft dead-ball rule: *ball visible and outside the pitch (with a margin) → dead*. Script: `ball_outside.py`.

> **Run this notebook on the VM only.** Once run, its outputs are derived from the competition match. Before a copy with outputs leaves the VM, clear it:
> `jupyter nbconvert --clear-output --inplace eda_ball_outside_pitch.ipynb`.

The script commands are **comments**: run them in a VM terminal from the team-repo root (or remove the `#` and prefix `!`). The cells below each command only read the summary CSVs the scripts wrote under `EDA_OUT` (default `~/eda_out`); none of them reads match data. Charts share one look from `plot_helpers.py`: one colour per set-piece `Type`, dark ink for observed values, a light grey band (p5–p95) with a dashed mean for the circular-shift null, and grey 'n < 5' stubs where there is too little to plot.

Definitions, defaults and CSV schemas: `eda/README.md`, section "Analyses".

In [ ]:
import os
import sys
from pathlib import Path

# The eda/ folder (Jupyter's working directory when the notebook is opened from there) and the
# script outputs. Override with EDA_DIR / EDA_OUT in the shell that starts Jupyter.
EDA_DIR = Path(os.environ.get("EDA_DIR", Path.cwd()))
EDA_OUT = Path(os.environ.get("EDA_OUT", Path.home() / "eda_out"))
sys.path.insert(0, str(EDA_DIR))

print("eda folder:", EDA_DIR, "(plot_helpers.py found)" if (EDA_DIR / "plot_helpers.py").is_file()
      else "(plot_helpers.py MISSING: set EDA_DIR)")
print("EDA_OUT   :", EDA_OUT)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import plot_helpers as ph

ph.style()

In [ ]:
CSV = "ball_outside_summary.csv"
bo = ph.load_summary(EDA_OUT, "ball_outside", CSV)
bo2 = ph.load_summary(EDA_OUT, "ball_outside_link2", CSV)    # sensitivity: --link-window 2

margins = ph.pick(bo, "params", "all", None, "margin")
MAIN = float(margins[margins.bin == "main"].value.iloc[0])
MARGINS = sorted(margins.value.astype(float))
W = ph.get(bo, "params", "all", "", "link_window")
K = ph.null_k(bo)
HAS_TAGS = len(ph.pick(bo, "link")) > 0


def sc(m, line):
    return "m{:g}:{}".format(m, line)


print("main margin {:g} m, all margins {}, link window {:g} s, null K = {}, tags: {}".format(
    MAIN, MARGINS, W, K, HAS_TAGS))

## 1. How often is the visible ball beyond the lines, and how much depends on the margin?

In [ ]:
# cd ~/more-than-a-hack
# PYTHONPATH="$PWD" python3 eda/ball_outside.py "$HOME/eda_out/ball_outside"

In [ ]:
goal = [ph.get(bo, "share", sc(m, "goal_line"), "", "share") for m in MARGINS]
touch = [ph.get(bo, "share", sc(m, "touchline"), "", "share") for m in MARGINS]
runs = [ph.get(bo, "follows", sc(m, "all"), "", "n_runs") for m in MARGINS]

fig, ax = plt.subplots(figsize=(6.8, 3.8))
x = np.arange(len(MARGINS))
ax.bar(x, goal, width=0.6, color=ph.REAL, edgecolor="white", lw=1.5, label="beyond a goal line")
ax.bar(x, touch, bottom=goal, width=0.6, color=ph.INK_3, edgecolor="white", lw=1.5,
       label="beyond a touchline")
top = max(g + t for g, t in zip(goal, touch)) or 1e-9
for i, (g, t, n) in enumerate(zip(goal, touch, runs)):
    ax.text(i, g + t + top * 0.02, "{:.2%}\nn = {:,} runs".format(g + t, int(n)), ha="center",
            va="bottom", fontsize=8, color=ph.INK_2)
ax.set_xticks(x, ["{:g} m{}".format(m, " (main)" if m == MAIN else "") for m in MARGINS])
ax.set_ylim(0, top * 1.4)
ax.set_xlabel("margin beyond the line (m)")
ax.set_ylabel("share of ball-visible graded frames")
ax.set_title("How often is the visible ball beyond the lines?")
ax.legend(loc="upper right")
plt.show()

display(pd.DataFrame({
    "margin (m)": MARGINS,
    "visible frames": [int(ph.get(bo, "share", sc(m, "all"), "", "visible_frames")) for m in MARGINS],
    "out: goal line": [int(ph.get(bo, "share", sc(m, "goal_line"), "", "out_frames")) for m in MARGINS],
    "out: touchline": [int(ph.get(bo, "share", sc(m, "touchline"), "", "out_frames")) for m in MARGINS],
    "runs": [int(n) for n in runs]}).set_index("margin (m)"))

**How to read this.** Each bar is the share of ball-visible graded frames on which the ball is beyond a line by more than the margin; dark is beyond a goal line, light beyond a touchline. The label gives the total share and the number of out-of-bounds runs. A large drop from 0 m to 1 m means most "out" frames are line jitter, not stoppages. A tiny share at the main margin means the tracker rarely follows the ball off the pitch, so clause a covers little time on its own and the out-then-missing pattern (section 3) matters more.

**Takeaway.** _to fill in after running_

## 2. Are out-of-bounds runs short blips or sustained stretches?

In [ ]:
# (same run as section 1)

In [ ]:
BINS = ["<0.5s", "0.5-2s", "2-5s", ">=5s"]
n_runs = [ph.get(bo, "runs", sc(MAIN, "all"), b, "n_runs") for b in BINS]
frames = [ph.get(bo, "runs", sc(MAIN, "all"), b, "frames") for b in BINS]
total = sum(frames) or 1

fig, ax = plt.subplots(figsize=(6.8, 3.6))
ph.bars_with_n(ax, BINS, [f / total for f in frames], n_runs, fmt="{:.0%}")
ax.set_xlabel("run length (s, half-open bins)")
ax.set_ylabel("share of out-of-bounds frames")
ax.set_title("Where does out-of-bounds time sit: blips or long runs? ({:g} m margin)".format(MAIN))
plt.show()

display(pd.DataFrame({line: [int(ph.get(bo, "runs", sc(MAIN, line), b, "n_runs")) for b in BINS]
                      for line in ("all", "goal_line", "touchline")}, index=BINS)
        .rename_axis("run length").add_prefix("runs: "))

**How to read this.** Bar height is the share of out-of-bounds frames in runs of that length; the label adds the number of runs. Bins with fewer than 5 runs are grey stubs. Mass in the short bins (< 0.5 s) points to jitter or a ball crossing and coming straight back, which clause a should not call dead; mass in runs of 2 s or more supports it.

**Takeaway.** _to fill in after running_

## 3. Right after the ball leaves the pitch, does it go missing or come back inside?

In [ ]:
# (same run as section 1)

In [ ]:
LINES = ["all", "goal_line", "touchline"]
OUTCOMES = [("missing", ph.REAL, "ball goes missing"), ("inside", ph.INK_3, "ball back inside"),
            ("half_end", ph.NO_TYPE, "half ends")]

fig, ax = plt.subplots(figsize=(7.2, 2.8))
for i, line in enumerate(LINES):
    n = ph.get(bo, "follows", sc(MAIN, line), "", "n_runs")
    left = 0.0
    for key, colour, label in OUTCOMES:
        share = ph.get(bo, "follows", sc(MAIN, line), key, "share")
        share = 0.0 if not np.isfinite(share) or n < ph.MIN_N else share
        ax.barh(i, share, left=left, height=0.6, color=colour, edgecolor="white", lw=1.5,
                label=label if i == 0 else None)
        if share >= 0.08:
            ax.text(left + share / 2, i, "{:.0%}".format(share), ha="center", va="center",
                    fontsize=8, color="white" if colour == ph.REAL else ph.REAL)
        left += share
    ax.text(1.01, i, "n < 5" if n < ph.MIN_N else "n = {:,}".format(int(n)), va="center",
            fontsize=8, color=ph.INK_2)
ax.set_yticks(range(len(LINES)), ["all runs", "goal line", "touchline"])
ax.invert_yaxis()
ax.set_xlim(0, 1.12)
ax.set_xlabel("share of out-of-bounds runs ({:g} m margin)".format(MAIN))
ax.set_title("After the ball leaves the pitch, does the tracker lose it?")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=3)
plt.show()

**How to read this.** Each row splits the out-of-bounds runs by what happens on the next frame: the ball goes missing (dark), is back inside the pitch (light), or the half ends. If most runs end with the ball going missing, a rule can carry "dead" from the out-of-bounds frame into the gap that follows, which is where most stoppage time would be. If most come straight back inside, the runs are mostly tracking noise near the line.

**Takeaway.** _to fill in after running_

## 4. Do out-of-bounds runs end near a set-piece start more often than chance?

In [ ]:
# (same run as section 1); sensitivity with a tighter window:
# PYTHONPATH="$PWD" python3 eda/ball_outside.py "$HOME/eda_out/ball_outside_link2" --link-window 2

In [ ]:
if not HAS_TAGS:
    print("no Smart Tagging file in this run: section skipped")
else:
    groups = ph.sort_groups(ph.pick(bo, "link", sc(MAIN, "all"), None, "real_share").bin.unique())
    fig, ax = plt.subplots(figsize=(7.2, 0.5 * len(groups) + 1.4))
    rows = []
    for i, g in enumerate(groups):
        def v(metric, df=bo):
            return ph.get(df, "link", sc(MAIN, "all"), g, metric)
        n = v("n_runs")
        if np.isfinite(v("null_p5")):
            ph.null_interval(ax, i, v("null_p5"), v("null_p95"), v("null_mean"))
        if n >= ph.MIN_N:
            ax.plot(v("real_share"), i, "o", ms=9, color=ph.type_colour(g), mec="white", mew=1.5,
                    zorder=3)
        ax.text(1.02, i, "n < 5" if n < ph.MIN_N else
                "{:.0%}  lift {:.1f}  n = {:,}".format(v("real_share"), v("lift"), int(n)),
                va="center", fontsize=8, color=ph.INK_2, transform=ax.get_yaxis_transform())
        rows.append({"group": ph.group_label(g), "runs": int(n), "linked": int(v("real_count")),
                     "share": v("real_share"), "null mean": v("null_mean"),
                     "lift": v("lift"), "p": ph.fmt_p(v("p_emp"), K),
                     "share (W = 2 s)": v("real_share", bo2) if bo2 is not None else np.nan})
    ax.set_yticks(range(len(groups)), [ph.group_label(g) for g in groups])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel("share of out-of-bounds runs with a start within ±{:g} s".format(W))
    ax.set_title("Out-of-bounds runs vs set-piece starts: real (dots) vs chance (grey)")
    plt.show()
    display(pd.DataFrame(rows).set_index("group").round(3))

**How to read this.** For each group (all set pieces, then each `Type` in its colour), the dot is the share of out-of-bounds runs (all lines, main margin) whose last frame is within ±W s of a start of that group. The grey band is the same share with the tags shifted circularly within each half (p5–p95), the dashed tick its mean. A dot well right of the band (lift ≫ 1, small p) means the ball leaving the pitch really precedes restarts, which supports clause a. Throw-ins should dominate touchline runs and corners/goal kicks goal-line runs (section 5). The table adds the share with a ±2 s window, if that run exists.

**Takeaway.** _to fill in after running_

## 5. Which restart follows each line?

In [ ]:
# (same run as section 1)

In [ ]:
if not HAS_TAGS:
    print("no Smart Tagging file in this run: section skipped")
else:
    ct = ph.pick(bo, "crosstab", None, None, "n_runs")
    lines = ["goal_line", "touchline"]
    types = ph.sort_groups([t for t in ct.bin.unique() if t not in ("none", "(no Type)")])
    types += [t for t in ("(no Type)", "none") if t in set(ct.bin)]
    fig, ax = plt.subplots(figsize=(7.2, 2.6))
    table = {}
    for i, line in enumerate(lines):
        counts = {t: ph.get(bo, "crosstab", sc(MAIN, line), t, "n_runs") for t in types}
        counts = {t: (0 if not np.isfinite(n) else int(n)) for t, n in counts.items()}
        table[line] = counts
        n = sum(counts.values())
        left = 0.0
        for t in types:
            share = counts[t] / n if n >= ph.MIN_N else 0.0
            ax.barh(i, share, left=left, height=0.6, color=ph.type_colour(t), edgecolor="white",
                    lw=1.5)
            if share >= 0.08:
                ax.text(left + share / 2, i, str(counts[t]), ha="center", va="center",
                        fontsize=8, color="white" if t not in ("none", "(no Type)") else ph.REAL)
            left += share
        ax.text(1.01, i, "n < 5" if n < ph.MIN_N else "n = {:,}".format(n), va="center",
                fontsize=8, color=ph.INK_2)
    ax.set_yticks(range(len(lines)), ["goal line", "touchline"])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("share of out-of-bounds runs ({:g} m margin)".format(MAIN))
    ax.set_title("Which restart starts within ±{:g} s of the ball leaving the pitch?".format(W))
    ph.type_legend(ax, [t for t in types if t not in ("none", "(no Type)")], extra=[("no set piece within W", ph.NO_TYPE)],
                   below=-0.3, ncol=3)
    plt.show()
    display(pd.DataFrame(table).T.rename_axis("line"))

**How to read this.** Each row splits the out-of-bounds runs over one kind of line by the `Type` of the nearest set-piece start within ±W s (colours as in every notebook; light grey = no set piece nearby). Segment labels are counts. The football expectation is throw-ins after touchline runs and corners or goal kicks after goal-line runs. A large light-grey share means many out-of-bounds runs have no tagged restart nearby: either tagging is thin there, or the runs are not real stoppages.

**Takeaway.** _to fill in after running_